# 0. Setup

In [ ]:
# COMMENT IN WHEN RUNNING FOR THE FIRST TIME
#  %pip install -r requirements.txt

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from tqdm import tqdm
import pennylane as qml
import numpy as np
import pandas as pd
import torch

seed = 2408
torch.manual_seed(seed)

# 1. The Wisconsin Diagnostic Breast Cancer Dataset

The WDBC dataset (Street, Wolberg & Mangasarian, 1993) contains **569 patients**. For each, a fine-needle aspirate (FNA) of a breast mass was digitised and the cell nuclei in the image were described by **10 properties**:

radius · texture · perimeter · area · smoothness · compactness · concavity · concave points · symmetry · fractal dimension

For each property three statistics are reported — the **mean**, the **standard error** and the **"worst"** (mean of the three largest values) — giving **30 real-valued features**. The target is the diagnosis: **malignant** or **benign**.

In [3]:
data = load_breast_cancer(as_frame=True)
X_df = data.data
# In scikit-learn: 0 = malignant, 1 = benign. We recode so that 1 = malignant (the "positive" finding).
y = (data.target == 0).astype(int).values
feature_names = list(X_df.columns)
class_names = {0: "benign", 1: "malignant"}

print(f"Patients: {X_df.shape[0]}, features: {X_df.shape[1]}")
print("Class counts:", {class_names[k]: int(v) for k, v in zip(*np.unique(y, return_counts=True))})
X_df.describe().T[["mean", "std", "min", "max"]].round(3)

Patients: 569, features: 30
Class counts: {'benign': 357, 'malignant': 212}


,mean,std,min,max
mean radius,14.127,3.524,6.981,28.110
mean texture,19.290,4.301,9.710,39.280
mean perimeter,91.969,24.299,43.790,188.500
mean area,654.889,351.914,143.500,2501.000
mean smoothness,0.096,0.014,0.053,0.163
mean compactness,0.104,0.053,0.019,0.345
mean concavity,0.089,0.080,0.000,0.427
mean concave points,0.049,0.039,0.000,0.201
mean symmetry,0.181,0.027,0.106,0.304
mean fractal dimension,0.063,0.007,0.050,0.097


# 3. Preprocessing

Now we have imported the dataset. However there are 2 important problems we need to tackle:

1. The dimensionality of the data is too high to simulate the quantum computer effectively on our Laptops
2. We need to translate the data to the language of the quantum computer, i.e. qubits

In [ ]:
#Number of PCA components
k = 4

In [ ]:
wdbc_dataset = load_breast_cancer()

# Pre-scaling of features since the PCA is sensitive to scaling
feature_scaler = MinMaxScaler()
features = feature_scaler.fit_transform(wdbc_dataset.data)

# using only label 0 (malignant); 1 = benign
mask = wdbc_dataset.target == 0
features = features[mask]
labels = torch.tensor(wdbc_dataset.target[mask])

# applying PCA
pca_model = PCA(n_components=k)
images_pca = pca_model.fit_transform(features)

## 3.1 Setting up the model 

In [ ]:
# Define some model parameters
n_data_qubits = k
n_gen_qubits = k
total_wires = 1 + n_data_qubits + n_gen_qubits

ancilla_wire = 0
data_start = 1
data_end = data_start + n_data_qubits - 1
gen_start = data_end + 1
gen_end = gen_start + n_gen_qubits - 1

Now we are going to set up the Quantum GAN. For this example we will use an easier version, where both of the network parts are quantum, but the Discriminator will be defined via 

In [ ]:
dev = qml.device("default.qubit", wires=total_wires)

# feature map
def angle_encoding(input_data: torch.Tensor):
    # use all data qubits 1..8
    x = torch.clamp(input_data, 0.0, 1.0 - 1e-7)
    encoded = 2 * torch.asin(torch.sqrt(x))  # now length 8

    qml.AngleEmbedding(
        features=encoded,
        wires=range(data_start, data_end + 1),  # 1..8
        rotation="Y",
    )


# generator
def q_generator(weights: torch.Tensor):
    # generator on qubits 9..16
    for w, qubit in enumerate(range(gen_start, gen_end + 1)):  # 9..16
        theta_x = weights[2 * w]
        theta_y = weights[2 * w + 1]
        qml.RX(theta_x, wires=qubit)
        qml.RY(theta_y, wires=qubit)


# discriminator (swap test)
def swap_test():
    qml.Hadamard(wires=ancilla_wire)
    for i, j in zip(range(data_start, data_end + 1), range(gen_start, gen_end + 1)):
        qml.CSWAP(wires=[ancilla_wire, i, j])
    qml.Hadamard(wires=ancilla_wire)


# final circuit
@qml.qnode(dev, interface="torch")
def circuit(input_data: torch.Tensor, weights: torch.Tensor):
    angle_encoding(input_data)  # now expects 8-dim vector
    q_generator(weights)        # 16 params
    swap_test()
    return qml.probs(wires=ancilla_wire)

Visualizing the circuit.

In [ ]:
fig, ax = qml.draw_mpl(circuit, style="black_white_dark")(torch.rand(k), torch.rand(2 * n_gen_qubits))
ax.set_title("IQGAN", fontsize=30, x=0.5, y=0.94)
fig.show()

## 3.2 Training Loop

In [ ]:
#####################  GENERATOR FUNCTION  ####################

@qml.qnode(dev, interface="torch")
def measuring_generator(weights: torch.Tensor):
    q_generator(weights)
    ms = []
    for w in range(gen_start, gen_end + 1):
        ms.append(qml.probs(wires=w))
    return tuple(ms)


#####################  TRANING PROCEDURE  #####################
t_dev = torch.device('cpu')
weights_generator = torch.rand(2 * n_gen_qubits, requires_grad=True, device=t_dev)

optimizer = torch.optim.Adam([weights_generator], lr=learning_rate)


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

# list for plotting the fidelity per epoch ("ot" -> over-time)
fidelity_ot = []

# list for generator loss per iteration (for plotting)
loss_per_iteration = []

# variables for storing the best generator weights
best_generator_weights = None
best_loss = float("inf")

In [ ]:
print("Starting training...")
for epoch in range(n_epochs):
    print(f'EPOCH {epoch + 1}:')

    for i, data in enumerate(tqdm(data_loader)):
        ########################
        # training generator
        ########################

        # loading training image and sending it to the device
        images = data[0].to(t_dev)  # data = (images, labels)

        # zero the gradients
        optimizer.zero_grad()

        temp_loss = 0.0
        temp_fidelity = 0.0

        # loop over samples in the batch
        for img in images:
            # circuit function returns probabilities of measuring either 0 or 1 on the ancilla q_bit.
            # probs[0] -> P(measure 0), probs[1] -> P(measure 1)
            probs = circuit(input_data=img, weights=weights_generator)

            # fidelity corresponds to the probability of measuring "0"
            temp_fidelity = temp_fidelity + probs[0]

            # generator loss corresponds to the probability of measuring "1"
            temp_loss = temp_loss + probs[1]

        # generator loss as average over batch
        gen_loss = temp_loss / len(images)

        # fidelity as average over batch
        fidelity = temp_fidelity / len(images)

        # backward propagation
        gen_loss.backward()

        # adjusting learning weights
        optimizer.step()

        # saving best weights so far
        if gen_loss.item() < best_loss:
            best_loss = gen_loss.item()
            best_generator_weights = weights_generator.detach().cpu().clone().numpy()

        # printing results every 40 iterations (or on the final batch)
        if i % 40 == 39 or i == len(data_loader) - 1:
            print(f'  batch {i + 1} loss: {gen_loss.item():.6f}  fidelity: {fidelity.item():.6f}')

            # keeping track of the fidelity for later plotting
            fidelity_ot.append(fidelity.item())
            loss_per_iteration.append(gen_loss.item())

    # advancing scheduler
    scheduler.step()

    # best fidelity so far (1 - best_loss, because loss ~ P(1), fidelity ~ P(0))
    print(f'  best fidelity so far: {1.0 - best_loss:.6f}')


print("Training complete!----------------------------------------------")



##################### VISUALIZE LOSS  #############

# 4. Generating a patient

In [ ]:
def generate_patient(weights: torch.Tensor):
    # exact generator output: feature i = probability of measuring 1 on generator qubit i
    with torch.no_grad():
        ms = measuring_generator(weights=weights)
        x = torch.stack([m[1] for m in ms]).unsqueeze(0).cpu().numpy()  # shape (1, k)

    # back from normalized PCA space to the 30 clinical features
    sample = pca_scaler.inverse_transform(x)
    features_01 = pca_model.inverse_transform(sample)
    return feature_scaler.inverse_transform(features_01)[0]


best_weights = torch.tensor(best_generator_weights, device=t_dev)
generated_patient = generate_patient(best_weights)

real_malignant = wdbc_dataset.data[wdbc_dataset.target == 0]

patient_table = pd.DataFrame(
    {
        "generated patient": generated_patient,
        "mean of real malignant patients": real_malignant.mean(axis=0),
        "std of real malignant patients": real_malignant.std(axis=0),
    },
    index=wdbc_dataset.feature_names,
)
patient_table.round(3)

# 5. Simulating (close to) real hardware with shot noise

In [ ]:
# generator circuit that returns raw measurement outcomes (bitstrings) instead of probabilities
@qml.qnode(dev, interface="torch")
def sampling_generator(weights: torch.Tensor):
    q_generator(weights)
    return qml.sample(wires=range(gen_start, gen_end + 1))


def generate_with_shots(weights: torch.Tensor, n_shots: int, n_patients: int):
    """Generates n_patients samples; each feature is estimated from n_shots measurements.
    Returns the samples in the normalized PCA space, shape (n_patients, k)."""
    shot_circuit = qml.set_shots(sampling_generator, shots=n_shots * n_patients)
    bits = np.asarray(shot_circuit(weights)).reshape(n_patients, n_shots, n_gen_qubits)
    return bits.mean(axis=1)


def to_clinical(x_pca_norm: np.ndarray):
    return feature_scaler.inverse_transform(pca_model.inverse_transform(pca_scaler.inverse_transform(x_pca_norm)))


# exact generator output in the normalized PCA space
with torch.no_grad():
    exact_x = torch.stack([m[1] for m in measuring_generator(weights=best_weights)]).cpu().numpy()

# one patient generated with 100 shots per qubit, compared to the exact output
shot_patient = to_clinical(generate_with_shots(best_weights, n_shots=100, n_patients=1))[0]
pd.DataFrame(
    {"exact generator (infinite shots)": generated_patient, "generated with 100 shots": shot_patient},
    index=wdbc_dataset.feature_names,
).round(3).head(10)


# 5.1 Spread over shot noise

In [ ]:
shot_list = [1, 10, 100, 1000, 10000]
n_patients = 200

fig, axes = plt.subplots(1, len(shot_list), figsize=(4 * len(shot_list), 4), sharex=True, sharey=True)
for ax, n_shots in zip(axes, shot_list):
    gen = generate_with_shots(best_weights, n_shots=n_shots, n_patients=n_patients)
    ax.scatter(norm_images_pca_np[:, 0], norm_images_pca_np[:, 1], s=10, c="lightgrey", label="real malignant patients")
    ax.scatter(gen[:, 0], gen[:, 1], s=10, alpha=0.5, c="tab:orange", label="generated")
    ax.scatter(exact_x[0], exact_x[1], marker="x", s=120, c="black", label="exact generator")
    ax.set_title(f"{n_shots} shot{'s' if n_shots > 1 else ''}")
    ax.set_xlabel("PC 1 (normalized)")
axes[0].set_ylabel("PC 2 (normalized)")
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()